# 02 — MCMC Diagnostics & Posterior Predictive

Load MCMC output from `outputs/mcmc/<run_id>.nc` and inspect convergence,
parameter recovery, and posterior predictive skill.

**Pre-requisites:**
1. `python config.py` — generate data
2. `python run_mcmc.py --run_id rwmh_v1` — run MCMC and save NetCDF

Change `RUN_ID` below to switch between saved runs.


In [ ]:
import notebook_env  # noqa: F401 — sets sys.path for config import
import numpy as np
import matplotlib.pyplot as plt
import arviz as az

from config import build_prior, OUTPUT_DIR, GROUND_TRUTH, DATA_DIR, SIGMA_OBS
from sipnet_calibration import default_base_params, posterior_predictive
from sipnet_calibration.plotting import fan_chart, plot_diagnostics
from pysipnet import SIPNETRunner, SIPNETModel
from pysipnet.climate import ClimateDrivers
from pysipnet.runner import ClimateStaging


In [ ]:
RUN_ID = "rwmh_v1"   # ← change this to load a different run

nc_path = OUTPUT_DIR / "mcmc" / f"{RUN_ID}.nc"
idata = az.from_netcdf(str(nc_path))
print(f"Loaded: {nc_path}")
print(idata)


In [ ]:
prior = build_prior()
param_names = list(prior.record_template.fields)

plot_diagnostics(idata, param_names=param_names, ground_truth=GROUND_TRUTH)
plt.show()


In [ ]:
# Posterior predictive check
# Reconstruct posterior chains from ArviZ InferenceData
# Shape: (n_chains, n_draws, n_params) from the raw RWMH flat array
# ArviZ stores flattened chains in idata.posterior as individual param DataArrays.
# We need the raw flat chain (pre-unflatten) — store it separately via run_mcmc.py
# OR re-derive from named params.

# Derive flat chains from named posterior params in idata
n_chains = idata.posterior.dims["chain"]
n_draws  = idata.posterior.dims["draw"]
n_params = prior.record_template.flat_size

# Build flat chains from the unflattened stored values
# (we'll let posterior_predictive handle this by passing the named arrays)

# Simpler: extract per-param arrays and pass directly to model
# Here we use a convenience path: build a (n_chains, n_draws, n_params) array
# by stacking named params in prior field order.
flat_chains = np.stack(
    [np.array(idata.posterior[name]) for name in param_names],
    axis=-1,
)  # shape: (n_chains, n_draws, n_params)

print(f"Flat chains shape: {flat_chains.shape}")

# Rebuild SIPNETModel
climate = ClimateDrivers.from_path(str(DATA_DIR / "climate.clim"))
sipnet_model = SIPNETModel(
    SIPNETRunner(climate_staging=ClimateStaging.SYMLINK),
    base_params=default_base_params(),
    base_climate=climate,
)

obs_nee = np.load(str(DATA_DIR / "obs_nee.npy"))
truth_nee = sipnet_model(**GROUND_TRUTH).nee().values


In [ ]:
# NOTE: flat_chains above is in NAMED (constrained) space, but posterior_predictive
# expects flat chains in the UNCONSTRAINED (prior) space for unflatten_value().
# Since ArviZ stores the constrained values, we skip unflatten and instead
# pass the model overrides directly.

# Custom posterior predictive (bypassing unflatten):
from pyens import EnsembleRunner, EnsembleSpec, Axis
from pyens.backends import LocalBackend
from pysipnet.ensemble import sipnet_member_fields

draws_flat = flat_chains.reshape(-1, flat_chains.shape[-1])  # (n_total, n_params)
rng = np.random.default_rng(1)
idx = rng.choice(len(draws_flat), size=min(200, len(draws_flat)), replace=False)
selected = draws_flat[idx]

param_arrays = {
    name: [float(selected[i, j]) for i in range(len(idx))]
    for j, name in enumerate(param_names)
}

members = Axis("member", size=len(idx))
spec = EnsembleSpec(inputs=sipnet_member_fields(members, **param_arrays))
runner = EnsembleRunner(sipnet_model, LocalBackend(n_workers=4))
result = runner.run(spec)

post_pred_nee = np.stack([r.output.nee().values for r in result.succeeded])
print(f"Posterior predictive shape: {post_pred_nee.shape}")


In [ ]:
t = np.arange(post_pred_nee.shape[1])

ax = fan_chart(
    post_pred_nee,
    t=t,
    obs=obs_nee,
    truth=truth_nee,
    ylabel="NEE (gC m$^{-2}$ per 3-hr step)",
    title=f"Posterior predictive check  [{RUN_ID}]",
    color="steelblue",
)
plt.tight_layout()
plt.show()


In [ ]:
# Marginal posterior histograms vs ground truth
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.ravel()

for i, name in enumerate(param_names):
    ax = axes[i]
    vals = np.array(idata.posterior[name]).ravel()
    ax.hist(vals, bins=50, density=True, alpha=0.7, color="steelblue")
    ax.axvline(GROUND_TRUTH[name], color="red", lw=2, label="truth")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("Value", fontsize=8)
    ax.legend(fontsize=7)

plt.suptitle(f"Posterior marginals  [{RUN_ID}]  (red = ground truth)")
plt.tight_layout()
plt.show()
